# RBAC Automation with `fiddler_utils`

This notebook demonstrates how to manage Fiddler's role-based access control
(RBAC) programmatically using the `fiddler_utils.rbac` subpackage.

It covers two complementary layers:

1. **Low-level managers** — `UserManager`, `TeamManager`, `ProjectRoleManager`
   for ad-hoc operations (list users, create a team, assign a role, etc.).
2. **Desired-state orchestrator** — `BulkRBACSync` for declaring the RBAC state
   you want and reconciling it against the live org in a single pass, with a
   safe dry-run preview and an opt-in apply.

## Use cases

* Onboarding a new team: create the team, add members, grant project access.
* Standing up a project: bulk-assign users and teams in one declarative spec.
* Drift detection: dry-run a spec to surface assignments that don't match policy.
* Audit trail: export a per-action CSV report after every sync.
* Periodic reconciliation: re-run the same spec on a schedule — the orchestrator
  is idempotent.

## Prerequisites

* Fiddler instance URL and an API token belonging to an **Org Admin** (RBAC
  endpoints require admin-level permissions).
* At least one project in your Fiddler org to assign roles against.
* (Optional) An existing user email to add to a demo team. If you skip this,
  the team-membership cells gracefully no-op.

## Safety notes

* The notebook defaults to **dry-run only**. No writes happen until you flip
  `APPLY = True` near the top.
* Cleanup at the end is **also gated** behind `CLEANUP = True` so re-running
  the notebook does not surprise-delete state.
* All resources the notebook creates use the `rbac_example_<suffix>` naming
  convention, so they cannot collide with real customer resources.
* Pruning flags on `TeamSpec` / `RBACSpec` default to `False` — the spec is
  strictly additive unless you opt in.


## Table of contents

1. [Setup](#Setup)
2. [Section 1: Low-level managers](#Section-1:-Low-level-managers)
   * 1.1 List users
   * 1.2 Find a user by email
   * 1.3 Create (or get) a team
   * 1.4 Add a member to the team
   * 1.5 Assign the team to a project
   * 1.6 Discover what is assignable
3. [Section 2: Bulk RBAC sync](#Section-2:-Bulk-RBAC-sync-(desired-state))
   * 2.1 Build a spec
   * 2.2 Dry-run preview
   * 2.3 Apply (gated)
   * 2.4 Re-run to demonstrate idempotency
4. [Section 3: Error-handling vignette](#Section-3:-Error-handling-vignette)
5. [Section 4: Cleanup (gated)](#Section-4:-Cleanup-(gated))
6. [Summary](#Summary)


---

## Setup


In [ ]:
# The RBAC subpackage ships in fiddler-utils v1.1.1+.
%pip install -q fiddler-client
%pip install -q "fiddler-utils @ git+https://github.com/fiddler-labs/fiddler-utils.git@v1.1.1"


### Imports

In [ ]:
import logging
import os
import uuid

import fiddler as fdl

from fiddler_utils import (
    # Low-level managers
    UserManager,
    TeamManager,
    ProjectRoleManager,
    # Spec dataclasses
    RBACSpec,
    TeamSpec,
    ProjectAssignmentSpec,
    # Enums
    ProjectRole,
    # Orchestrator
    BulkRBACSync,
    # Connection
    get_or_init,
)
from fiddler_utils.exceptions import BulkRBACSyncError
from fiddler_utils.rbac.exceptions import (
    OrgAdminAssignmentError,
    ProjectRoleConflictError,
    TeamNotFoundError,
    UserNotFoundError,
)

# Surface BulkRBACSync's per-phase INFO logs so dry-runs read like a plan.
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(name)s] %(levelname)s %(message)s")
logging.getLogger("fiddler_utils").setLevel(logging.INFO)

print(f"✓ fiddler-client {fdl.__version__}")


### Configuration

Set the four constants below (or export the matching `FIDDLER_*` env vars).
The user-related cells gracefully skip if `TEST_USER_EMAIL` is left blank.


In [ ]:
URL = os.environ.get("FIDDLER_URL", "https://your-org.cloud.fiddler.ai")
TOKEN = os.environ.get("FIDDLER_TOKEN", "YOUR_FIDDLER_TOKEN")
PROJECT_NAME = os.environ.get("FIDDLER_PROJECT", "")  # required for project-role cells
TEST_USER_EMAIL = os.environ.get("FIDDLER_TEST_USER_EMAIL", "")  # optional

# Sandboxed naming so this notebook can never collide with real teams/projects.
# Re-running the notebook reuses the same suffix during one session.
SUFFIX = uuid.uuid4().hex[:8]
SANDBOX_TEAM_A = f"rbac_example_team_a_{SUFFIX}"
SANDBOX_TEAM_B = f"rbac_example_team_b_{SUFFIX}"

# Safety flags. Flip to True only when you are ready to write to your Fiddler
# instance. Apply changes are gated so a misclick on "Run All" stays harmless.
APPLY = False    # gate writes in Section 1 + Section 2
CLEANUP = False  # gate the teardown in Section 4

print(f"Sandbox team A: {SANDBOX_TEAM_A}")
print(f"Sandbox team B: {SANDBOX_TEAM_B}")
print(f"APPLY={APPLY}, CLEANUP={CLEANUP}")


### Connect

`get_or_init` returns the existing SDK connection if there is one, or
calls `fdl.init(...)` to create one. The RBAC managers piggyback on the
SDK's connection, so no extra wiring is needed.


In [ ]:
get_or_init(url=URL, token=TOKEN)
print(f"✓ Connected to {URL}")


---

## Section 1: Low-level managers

The three RBAC managers wrap Fiddler's V3 endpoints with a small,
self-explanatory surface area. Each manager is cheap to instantiate; create
one per logical unit of work.


In [ ]:
users = UserManager()
teams = TeamManager()
project_roles = ProjectRoleManager()


### 1.1 List users

`UserManager.list()` paginates `/v3/users` for you and returns
`UserRecord` instances. Pass `limit=` if you only need the first N.


In [ ]:
all_users = users.list(limit=10)
print(f"Returned {len(all_users)} users (capped at 10).")
for u in all_users[:5]:
    print(f"  - {u.email!r:40s}  id={u.id}")


### 1.2 Find a user by email

`find_by_email` is case-insensitive and caches lookups on the manager
instance, so repeated calls don't re-paginate.


In [ ]:
found_user = None
if TEST_USER_EMAIL:
    found_user = users.find_by_email(TEST_USER_EMAIL)
    if found_user:
        print(f"✓ found {found_user.email!r} (id={found_user.id})")
    else:
        print(f"⚠ no user with email {TEST_USER_EMAIL!r} — membership cells will skip")
else:
    print("FIDDLER_TEST_USER_EMAIL not set — skipping user lookup")


### 1.3 Create (or get) a team

`get_or_create` is the safe primitive for ensuring a team exists: it
returns `(team, created)` and never errors if the team already exists.
The notebook only writes when `APPLY = True`.


In [ ]:
sandbox_team = None
if APPLY:
    sandbox_team, created = teams.get_or_create(SANDBOX_TEAM_A)
    verb = "Created" if created else "Found existing"
    print(f"{verb} team {sandbox_team.name!r} (id={sandbox_team.id})")
else:
    print(f"[dry-run] would get_or_create team {SANDBOX_TEAM_A!r}")


### 1.4 Add a member to the team

`add_member` returns a `TeamMembershipRecord` whose `id` you keep around
in case you later want to remove the membership.


In [ ]:
sandbox_membership = None
if APPLY and sandbox_team is not None and found_user is not None:
    sandbox_membership = teams.add_member(sandbox_team.id, found_user.id)
    print(f"✓ added {found_user.email} to {sandbox_team.name} (membership id={sandbox_membership.id})")
elif not APPLY:
    print("[dry-run] would add user to team")
else:
    print("Skipping: APPLY=True requires both a created team and a resolved test user")


### 1.5 Assign the team to a project

`assign_team` posts to `/v3/project-roles` with a team subject. Use
`assign_user` for user-targeted assignments. Both accept either a
`ProjectRole` enum or its string value.


In [ ]:
sandbox_role = None
if APPLY and sandbox_team is not None and PROJECT_NAME:
    project = fdl.Project.from_name(name=PROJECT_NAME)
    try:
        sandbox_role = project_roles.assign_team(
            project_id=str(project.id),
            team_id=sandbox_team.id,
            role=ProjectRole.WRITER,
        )
        print(
            f"✓ assigned team {sandbox_team.name!r} to project "
            f"{PROJECT_NAME!r} as {ProjectRole.WRITER.value} (id={sandbox_role.id})"
        )
    except ProjectRoleConflictError:
        print("Team was already assigned to this project (no-op).")
elif not APPLY:
    print(f"[dry-run] would assign team {SANDBOX_TEAM_A!r} to project {PROJECT_NAME!r} as Project Writer")
else:
    print("Skipping: APPLY=True requires a created team and FIDDLER_PROJECT to be set")


### 1.6 Discover what is assignable

`list_assignable_users` and `list_assignable_teams` hit the
`/v3/project-roles/assignable-*` endpoints. With no `project_id`,
they return everyone in the org; pass `project_id=` to restrict to
subjects not yet assigned to that specific project.


In [ ]:
assignable_users = project_roles.list_assignable_users(limit=5)
print(f"Org-wide assignable users (first 5 of however many):")
for u in assignable_users:
    print(f"  - {u.email!r:40s}  id={u.id}")

assignable_teams = project_roles.list_assignable_teams(limit=5)
print(f"\nOrg-wide assignable teams (first 5):")
for t in assignable_teams:
    print(f"  - {t.name!r:30s}  id={t.id}")


---

## Section 2: Bulk RBAC sync (desired-state)

`BulkRBACSync` takes an `RBACSpec` describing the teams, members, and
project-role assignments you want, then reconciles the live org in three
phases:

1. **Phase 1** — ensure each team exists and reconcile its members.
2. **Phase 2** — create / update each project-role assignment.
3. **Phase 3** — opt-in pruning (only runs when `prune_extra_*` flags are set).

The orchestrator is **additive by default**. Pruning is opt-in at three
granularities: `TeamSpec.prune_extra_members`, `RBACSpec.prune_extra_teams`,
and `RBACSpec.prune_extra_project_roles`. Even when enabled,
`prune_extra_project_roles` only touches projects mentioned in the spec —
it never operates org-wide.

### 2.1 Build a spec

We declare two sandboxed teams and one team-targeted project assignment.
Members are optional — the spec is valid with an empty `members=()` tuple,
which is handy for "ensure the team exists" use cases.


In [ ]:
members_for_team_a = (TEST_USER_EMAIL,) if TEST_USER_EMAIL else ()

spec = RBACSpec(
    teams=(
        TeamSpec(
            name=SANDBOX_TEAM_A,
            members=members_for_team_a,
            # Strictly additive: leave any existing members alone.
            prune_extra_members=False,
        ),
        TeamSpec(
            name=SANDBOX_TEAM_B,
            members=(),
        ),
    ),
    project_assignments=(
        ProjectAssignmentSpec(
            project=PROJECT_NAME,
            role=ProjectRole.WRITER,
            team_name=SANDBOX_TEAM_A,
        ),
        ProjectAssignmentSpec(
            project=PROJECT_NAME,
            role=ProjectRole.VIEWER,
            team_name=SANDBOX_TEAM_B,
        ),
    ) if PROJECT_NAME else (),
    # Org-level prune flags also default to False.
    prune_extra_teams=False,
    prune_extra_project_roles=False,
)

print(f"Spec has {len(spec.teams)} team(s) and {len(spec.project_assignments)} project assignment(s)")


### 2.2 Dry-run preview

Always dry-run first. The result object carries counters for every
category and a list of `(subject_label, error_message)` tuples. The
orchestrator's `print_report` formats it for humans.


In [ ]:
sync = BulkRBACSync(spec, on_missing_user="warn")
plan = sync.run(dry_run=True)
sync.print_report(plan)


You can also export a per-action CSV report — handy for change-management
audits.


In [ ]:
csv_path = f"rbac_sync_dryrun_{SUFFIX}.csv"
sync.export_report_csv(plan, csv_path)
print(f"Wrote dry-run report to {csv_path}")


### 2.3 Apply (gated)

Only runs when `APPLY = True`. The applied report is the same shape as
the dry-run report, with non-zero `*_created` / `*_added` counters.


In [ ]:
if APPLY:
    print("=== Applying changes ===")
    result = sync.run(dry_run=False)
    sync.print_report(result)
    sync.export_report_csv(result, f"rbac_sync_applied_{SUFFIX}.csv")
else:
    print("APPLY=False — skipping write. Re-run with APPLY=True to push the spec.")
    result = None


### 2.4 Re-run to demonstrate idempotency

Re-running the same spec against the now-converged state should report
zero `total_changes`. This is the property that makes the orchestrator
safe for scheduled / repeated runs.


In [ ]:
if APPLY:
    second = sync.run(dry_run=True)
    print(f"Second dry-run total_changes={second.total_changes} (expected 0 for an idempotent spec)")
    sync.print_report(second)
else:
    print("Skipped: idempotency check only meaningful after an APPLY=True run.")


---

## Section 3: Error-handling vignette

Two safety knobs cover the common edge cases you hit when running RBAC
syncs from a CSV / spreadsheet of asks:

* **`on_missing_user`** — what to do when an email in the spec doesn't
  resolve to a Fiddler user. `"warn"` (default) logs and skips the
  downstream operation; `"skip"` is the same at debug level; `"invite"`
  calls `UserManager.invite()` (the user becomes assignable on the *next*
  sync, after they sign up); `"raise"` aborts with `BulkRBACSyncError`.
* **`on_org_admin_assignment`** — Org Admins implicitly have Project Admin
  on every project, so the V3 API rejects explicit project-role
  assignments for them with a 422. `"warn"` (default) counts the rejection
  as `project_roles_unchanged`; `"skip"` is the quiet variant; `"raise"`
  surfaces it as a hard failure.

Below, we build a spec that intentionally references an email guaranteed
not to exist, and let `BulkRBACSync` show how it handles it.


In [ ]:
# We reuse SANDBOX_TEAM_A on purpose so the spec stays inside the sandbox
# namespace; only the bogus email is the variable under test.
bogus_spec = RBACSpec(
    teams=(
        TeamSpec(
            name=SANDBOX_TEAM_A,
            members=(f"does-not-exist-{SUFFIX}@example.invalid",),
        ),
    ),
)

sync_warn = BulkRBACSync(bogus_spec, on_missing_user="warn")
warn_result = sync_warn.run(dry_run=True)
sync_warn.print_report(warn_result)
print(f"\nmembers_skipped_missing_user = {warn_result.members_skipped_missing_user}")


Same spec, this time with `on_missing_user="raise"` — useful when you
want a CI pipeline to fail loudly on data-quality issues in the input.


In [ ]:
sync_strict = BulkRBACSync(bogus_spec, on_missing_user="raise")
try:
    sync_strict.run(dry_run=True)
except BulkRBACSyncError as exc:
    print(f"✓ got expected BulkRBACSyncError: {exc}")


### `on_org_admin_assignment` in practice

Org Admins implicitly have Project Admin on every project, so any explicit
project-role assignment for them is rejected by the V3 API with a 422.
The orchestrator catches that and folds it into `project_roles_unchanged`
so a routine sync isn't derailed by a spec line that names an Org Admin.

We can't safely *trigger* the 422 in a sandbox demo (it requires assigning
an actual Org Admin to a project, which is a real write), but we can show
how the knob is wired up. Below we instantiate a `BulkRBACSync` with
`on_org_admin_assignment="warn"` (the default), then re-instantiate with
`"raise"` so you can see how a CI pipeline would surface the rejection.


In [ ]:
demo_assignment_spec = RBACSpec(
    teams=(),
    project_assignments=(
        ProjectAssignmentSpec(
            project=PROJECT_NAME or "rbac_example_placeholder",
            role=ProjectRole.ADMIN,
            user_email=TEST_USER_EMAIL or f"someone-{SUFFIX}@example.invalid",
        ),
    ),
)

# Default behavior: log + count as project_roles_unchanged.
sync_warn_admin = BulkRBACSync(
    demo_assignment_spec,
    on_missing_user="warn",       # in case TEST_USER_EMAIL is unset
    on_org_admin_assignment="warn",
)
print(f"✓ BulkRBACSync configured with on_org_admin_assignment={sync_warn_admin.on_org_admin_assignment!r}")

# CI-friendly behavior: surface the rejection as BulkRBACSyncError.
sync_strict_admin = BulkRBACSync(
    demo_assignment_spec,
    on_missing_user="warn",
    on_org_admin_assignment="raise",
)
print(f"✓ BulkRBACSync configured with on_org_admin_assignment={sync_strict_admin.on_org_admin_assignment!r}")

# Knob validation runs at construction time.
try:
    BulkRBACSync(demo_assignment_spec, on_org_admin_assignment="not-a-thing")
except ValueError as exc:
    print(f"✓ invalid value rejected: {exc}")


---

## Section 4: Cleanup (gated)

Tear down everything the notebook created so re-running the example
doesn't leave junk in your Fiddler instance. Order matters: drop the
project-role first, then the team membership, then the team itself.

Set `CLEANUP = True` near the top of the notebook to enable.


In [ ]:
def _safe(label, fn, *args, **kwargs):
    """Run fn and print success / not-found / error without aborting cleanup."""
    try:
        fn(*args, **kwargs)
        print(f"  ✓ {label}")
    except (TeamNotFoundError, UserNotFoundError) as exc:
        print(f"  — {label}: not found ({exc})")
    except Exception as exc:
        print(f"  ✗ {label}: {type(exc).__name__}: {exc}")


SANDBOX_TEAM_NAMES = {SANDBOX_TEAM_A, SANDBOX_TEAM_B}

if CLEANUP:
    print("Cleaning up sandboxed RBAC state...")

    # 1. Project-role rows targeting any sandbox team on the project we
    #    touched. Done up-front — covers BOTH the row created in 1.5
    #    AND any rows created by Section 2's BulkRBACSync. We iterate the
    #    live list rather than relying on captured ids so we catch rows
    #    that were created in earlier kernel sessions.
    if PROJECT_NAME:
        try:
            project = fdl.Project.from_name(name=PROJECT_NAME)
            existing = project_roles.list(str(project.id))
            for role in existing:
                if role.subject_type != "team" or role.team is None:
                    continue
                if role.team.name not in SANDBOX_TEAM_NAMES:
                    continue
                _safe(
                    f"remove project-role {role.id} ({role.team.name!r} → {PROJECT_NAME!r})",
                    project_roles.remove,
                    role.id,
                )
        except Exception as exc:
            print(f"  ✗ list project-roles for {PROJECT_NAME!r}: "
                  f"{type(exc).__name__}: {exc}")

    # 2. Both sandbox teams. For each one, explicitly drop every membership
    #    before deleting the team — we don't rely on undocumented cascade
    #    behavior. Captures memberships added both by 1.4 (`sandbox_membership`)
    #    and by the bulk sync in Section 2 (which we never captured ids for).
    live_teams = {t.name: t for t in teams.list()}
    for name in (SANDBOX_TEAM_A, SANDBOX_TEAM_B):
        team_obj = live_teams.get(name)
        if team_obj is None:
            print(f"  — team {name!r}: not found")
            continue
        try:
            members = teams.list_members(team_obj.id)
        except Exception as exc:
            members = []
            print(f"  ✗ list members of {name!r}: {type(exc).__name__}: {exc}")
        for m in members:
            _safe(
                f"remove membership {m.id} from team {name!r}",
                teams.remove_member,
                m.id,
            )
        _safe(f"delete team {name!r}", teams.delete, team_obj.id)

    print("Cleanup complete.")
else:
    print("CLEANUP=False — leaving sandbox state in place. "
          "Set CLEANUP=True to tear down.")


---

## Summary

You've now seen both layers of the `fiddler_utils.rbac` API:

| Need | Use |
|---|---|
| One-off operation (create a team, add a user, grant a role) | A manager method directly |
| Reconcile a desired RBAC state from a CSV / config file | `BulkRBACSync` with an `RBACSpec` |
| Drift detection without writes | `BulkRBACSync(...).run(dry_run=True)` |
| CI / scheduled enforcement | `BulkRBACSync` + `on_error="raise"` + CSV report |
| Onboarding flow | `on_missing_user="invite"` |
